# Solar Thermal Tutorial in pandaprosumer
Example prepared by:  
Carl Ritzenhoff, carl.ritzenhoff@uni-kassel.de

## DESCRIPTION

This example describes how to create a single **solar thermal plant element** in pandaprosumer and connect it to a single consumer.  
The solar thermal plant receives solar radiation data from an external Excel file, while the consumer demand is defined as a constant profile.  
The tutorial demonstrates how to:
- Load time‑dependent input data from Excel into a pandas DataFrame
- Create a prosumer container and define a simulation period
- Add a SolarThermalController and a HeatDemandController
- Map the input/output parameters between controllers
- Run the time series simulation and inspect results



## Glossary

- **Network**: configuration of connected energy sources and consumers  
- **Element**: a single energy source or consumer  
- **Container**: pandaprosumer data structure that contains element data  
- **Controller**: logic of an element that defines behaviour and limits  
- **ConstProfileController**: general controller that manages external time‑dependent data  
- **Mapping**: connection between two elements, coupling outputs to inputs  

---


## Network design philosophy

In pandaprosumer, each component is represented by a network element with:
- a **container** (static data, configuration),
- a **controller** (dynamic behaviour),
- and **mappings** (connections between elements).

The ConstProfileController distributes external time‑dependent input data to relevant element controllers for each time step.


## Creating a network
If we are not in pandaprosumer parent directory, we should add it to the path so that the program knows where to find the necessary functions:

In [1]:
import sys
import os

current_directory = os.getcwd()
parent_directory = os.path.dirname(current_directory)
sys.path.append(parent_directory)

### 1. Input data
First let's import libraries required for data management.

In [2]:
import pandas as pd
from pandapower.timeseries.data_sources.frame_data import DFData

We use the default parameters for the sola thermal plant, so the dictionary stays empty

In [3]:
st_params = {}

We define the period of the analysis by setting its start and end, which have the form "YYYY-MM-DD HH:MM:SS". The time resolution is given in [s]. 

In [4]:
start = '2005-01-01 00:30:00'
end = '2005-01-05 00:29:59'
resol = 3600        # resolution in seconds (1h)
frequency = '60min'

Now we import our time-dependent data and transform it into an appropriate DFData object. DFData is a pandaprosumer object that stores all data of an individual element. The DFData object is limited by the duration of the analysis defined above. As well we define a demand.

In [5]:
data = pd.read_excel('data/senergy_nets_example_solar_thermal.xlsx')
data = data.iloc[3000:3096].copy()
data["q_demand_kw"] = 5
data["time"] = pd.to_datetime(data["time"], format="%Y%m%d:%H%M")
data.set_index("time", inplace=True)


dur = pd.date_range(start=start, end=end, freq=frequency, tz='utc')
data.index = dur
data_source = DFData(data)

print(list(data.columns))
data.head()

['Beam Solar Radiation [W/m2]', 'Diffuse Solar Radiation [W/m2]', 'Ground Solar Radiation [W/m2]', 'Radiation incidence angle [deg]', 'Ambient temperature [C]', 'Inlet temperature [C]', 'Inlet mass flow rate [kg/h]', 'q_demand_kw']


,Beam Solar Radiation [W/m2],Diffuse Solar Radiation [W/m2],Ground Solar Radiation [W/m2],Radiation incidence angle [deg],Ambient temperature [C],Inlet temperature [C],Inlet mass flow rate [kg/h],q_demand_kw
2005-01-01 00:30:00+00:00,0.0,0.00,0.00,0.00,1.00,10.0,0,5
2005-01-01 01:30:00+00:00,0.0,0.00,0.00,0.00,9.82,10.0,0,5
2005-01-01 02:30:00+00:00,0.0,0.00,0.00,0.00,0.90,10.0,0,5
2005-01-01 03:30:00+00:00,0.0,0.00,0.00,0.00,0.85,10.0,0,5
2005-01-01 04:30:00+00:00,0.0,22.01,0.53,4.92,8.38,10.0,0,5


### Creating network elements

In this example, the network is made up of two elements: an energy source and an energy consumer. 
The source is represented by a single solar thermal element and the consumer is modelled by a single heat demand element.

First we define an empty prosumer container object and the period. Each element of the network has its own container, which is later filled with data and results

In [6]:
from pandaprosumer.create import create_empty_prosumer_container
from pandaprosumer.create import create_period

prosumer = create_empty_prosumer_container()
period = create_period(prosumer, resol, start, end, 'utc', 'default')

#### General Element

In a pandaprosumer network, the first element is a general controller (Const Profile controller). It reads time-dependent input data (input_params) and sends it to other elements of the network (output_params). The element's data is stored in the ConstProfileControllerData class. The controller (ConstProfileController) for this element is created with the create_controlled_const_profile function. At this point, we pass to the general controller element the previously created prosumer container, titles of data columns (input_params) in the input file (Excel file, in this case) and the coresponding names of output columns (output_params), the period of the analysis and the time-dependent data in the DFData object

In [7]:
from pandaprosumer.create_controlled import create_controlled_const_profile

input_params = [
    'Beam Solar Radiation [W/m2]',
    'Diffuse Solar Radiation [W/m2]',
    'Ground Solar Radiation [W/m2]',
    'Radiation incidence angle [deg]',
    'Ambient temperature [C]',
    'Inlet temperature [C]',
    'Inlet mass flow rate [kg/h]',
    'q_demand_kw'
]

result_params = [
    "beam_solar_radiation_cp",
    "diffuse_solar_radiation_cp",
    "ground_solar_radiation_cp",
    "radiation_incidence_angle_cp",
    "ambient_temperature_cp",
    "inlet_temperature_cp",
    "inlet_mass_flow_rate_cp",
    "q_demand_kw_cp"
]

cp_index = create_controlled_const_profile(
    prosumer, input_params, result_params, data_source, period
)

#### 2.2 Solar Thermal Element

We define the solar thermal element to which we pass the prosumer container and the data that defines the solar thermal instance.

In [8]:
from pandaprosumer.create_controlled import create_controlled_solar_thermal

st_index = create_controlled_solar_thermal(
    prosumer, name="solar_thermal_plant", level=1, order=0, **st_params
)




#### 2.3 Heat Demand Element

We define the heat demand element to which we pass the prosumer container and the data that defines the heat demand instance.

In [9]:
from pandaprosumer.create_controlled import create_controlled_heat_demand

hd_index = create_controlled_heat_demand(
    prosumer,
    level=1,
    order=1,
    t_in_set_c=30,
    t_out_set_c=25,
)

### 3 - Creating connections (mappings) between controllers:

For each controller we define how it is connected to other controllers. In this case we use Generic Mapping. The main parameter for the map is the flow of thermal energy (energy_gain_W): the output energy flow of one element is linked with the input energy flow of the connected element. We usa a conversion function dor mapping the energy gin of the solar therml to the received thermal energy of the demand to meet the mapped unit.

In [10]:
from pandaprosumer.mapping import GenericMapping

GenericMapping(
    prosumer,
    initiator_id=cp_index,
    initiator_column=[
        "beam_solar_radiation_cp",
        "diffuse_solar_radiation_cp",
        "ground_solar_radiation_cp",
        "radiation_incidence_angle_cp",
        "ambient_temperature_cp",
        "inlet_temperature_cp",
        "inlet_mass_flow_rate_cp"
    ],
    responder_id=st_index,
    responder_column=[
        'beam_solar_radiation_w_m2',
        'diffuse_solar_radiation_w_m2',
        'ground_solar_radiation_w_m2',
        'radiation_incidence_angle_deg',
        'ambient_temperature_C',
        'inlet_temperature_C',
        'inlet_mass_flow_rate_kg_h',
    ]
)

GenericMapping(
    container=prosumer,
    initiator_id=cp_index,
    initiator_column="q_demand_kw_cp",
    responder_id=hd_index,
    responder_column="q_demand_kw",
)

GenericMapping(
    container=prosumer,
    initiator_id=st_index,
    initiator_column="energy_gain_W",
    responder_id=hd_index,
    responder_column="q_received_kw",
    order=0,
    conversion_function=lambda x: x / 1000,
)


### 4. RUNNING THE ANALYSIS:

In [11]:
from pandaprosumer.run_time_series import run_timeseries

run_timeseries(prosumer)

100%|██████████| 96/96 [00:00<00:00, 151.22it/s]


### 5. Results

In [12]:
results_df = prosumer.time_series.data_source.iloc[1].df
results_df.head(20)

,q_received_kw,q_uncovered_kw,mdot_kg_per_s,t_in_c,t_out_c
2005-01-01 00:30:00+00:00,0.000000,5.000000,0.0,0.0,0.0
2005-01-01 01:30:00+00:00,0.000000,5.000000,0.0,0.0,0.0
2005-01-01 02:30:00+00:00,0.000000,5.000000,0.0,0.0,0.0
2005-01-01 03:30:00+00:00,0.000000,5.000000,0.0,0.0,0.0
2005-01-01 04:30:00+00:00,0.000000,5.000000,0.0,0.0,0.0
2005-01-01 05:30:00+00:00,0.000000,5.000000,0.0,0.0,0.0
2005-01-01 06:30:00+00:00,0.000000,5.000000,0.0,0.0,0.0
2005-01-01 07:30:00+00:00,4.669914,0.330086,0.0,0.0,0.0
2005-01-01 08:30:00+00:00,5.064609,-0.064609,0.0,0.0,0.0
2005-01-01 09:30:00+00:00,5.072683,-0.072683,0.0,0.0,0.0
